Steps:
1. Load data from CSV and preprocess (remove stopwords) 
2. Find out vocabulary i.e unique words and assign numbers to each word 
3. Convert each word to number 
4. Find out the largest sequence and make all inputs the same length (using padding)
5. Define Dataset and Dataloader classes
6. LSTM model
7. Training code based on epoch and batch size, calculate loss 
8. Predict function 
9. Calculate metrics like accuracy, precision, recall etc 
10. Try it on a user review input

In [165]:
import torch 
import torch.nn as nn
import string
import pandas as pd
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
from collections import Counter
from helpers import word_to_num 
from torch.utils.data import Dataset, DataLoader


In [166]:
# Step 1: Load data from CSV and preprocess 

# Load CSV data
df = pd.read_csv("IMDB Dataset.csv")
df.head()

# Get stopwords list
stopwords_list = stopwords.words('english')

reviews_list = df.iloc[:, 0]
feedback_list = df.iloc[:, 1]

lemmatizer = WordNetLemmatizer()

# Preprocess sentence - remove punctuation, stopwords and convert to lowercase
clean_sentence = []
unique_words = set()
for sentence in reviews_list:
    words = word_tokenize(sentence)
    allowed_words = []
    for word in words:
        if word not in stopwords_list and word not in string.punctuation:
            allowed_words.append(lemmatizer.lemmatize(word.lower()))
    unique_words = unique_words.union(set(allowed_words))    
    clean_sentence.append(' '.join(allowed_words))

clean_sentence
unique_words



{'pityful',
 'progresses.',
 'instant',
 'anzac',
 'restaraunt',
 '13.',
 'rarely',
 'commitophobe',
 'original.the',
 'assessing',
 'realness',
 'shrimp',
 'malarkey.',
 'pressured',
 'scary.',
 'sagacious',
 'ministro',
 'madness',
 'mew',
 'nuke',
 'footnote',
 'pallette',
 'cinema.',
 'fetus',
 'eventually-trademarked',
 'bernier',
 'traversing',
 'garofalo',
 'arrives.',
 'miraculous',
 'indy',
 'giubba',
 'bellicose',
 'milligan',
 'predisposed',
 'skews',
 'brotherhood',
 '1907',
 'heartbreakng',
 'uneventful',
 '10-dimensional',
 'woah',
 '10-page',
 'intestine',
 'largesse',
 'off-springs',
 '7.5/10',
 'store',
 'reed',
 'biggies',
 "interesting'\x97and",
 'spectacle.the',
 'instrumentalist',
 'smiling',
 'cabourg',
 'romance.',
 'octopussy',
 'alcoholic-but-kind-hearted',
 'reflective',
 'barfed',
 'turbidity',
 'difficulty.',
 'naturalistic',
 "'happy",
 'mendoza',
 'atrocious.there',
 'emory',
 'life.i',
 'pitted',
 'errors.',
 'sarcasm',
 'law-breaking',
 'plus.',
 'cheape

In [167]:
# Step 2: Find out vocabulary i.e unique words and assign numbers to each word 
vocab = {'<UNK>': 0}
count = 1
for w in unique_words:
    vocab[w] = count 
    count += 1

In [168]:
# Step 3. Convert each word to number
reviews_int = []
seq_len = []
for sent in clean_sentence:
    sent_arr = sent.split(' ')
    for i, w in enumerate(sent_arr):
        sent_arr[i] = word_to_num(w, vocab)
    seq_len.append(len(sent_arr))
    reviews_int.append(sent_arr)

acceptable_len = max(seq_len)

print(reviews_int[:5])
print(acceptable_len)

[[48108, 8602, 18142, 9713, 35330, 35178, 38687, 48592, 26247, 46323, 40926, 25645, 26311, 39276, 31213, 31213, 17487, 27126, 11975, 48627, 35178, 11836, 24833, 11405, 10557, 24232, 40926, 19733, 1961, 49115, 45722, 10109, 3578, 46163, 45474, 45722, 14465, 1322, 49618, 35300, 18337, 10557, 48928, 25604, 41904, 15543, 33697, 31213, 31213, 48928, 45726, 35178, 48207, 30606, 29179, 36147, 5703, 11018, 8390, 48928, 26683, 37016, 37267, 18477, 13432, 16450, 3718, 7105, 14350, 6489, 11707, 23629, 40640, 11883, 8647, 14757, 18477, 45750, 38239, 11605, 30017, 30864, 24920, 42386, 18627, 46609, 12454, 1043, 45494, 40461, 1412, 33420, 23290, 11953, 42380, 45844, 45680, 26906, 31213, 31213, 27595, 39490, 36241, 38939, 9335, 45722, 181, 34876, 1961, 45722, 39490, 31711, 12327, 16316, 34336, 41589, 49078, 36295, 15205, 16316, 12516, 16316, 13226, 18686, 35178, 31711, 5765, 25270, 17487, 27126, 38687, 27595, 24211, 15669, 48627, 17331, 11808, 27595, 38104, 31711, 36241, 27595, 5349, 27595, 7933, 275

In [169]:
# Step 4. Find out the largest sequence and make all inputs the same length (using padding)
for i, rev in enumerate(reviews_int):
    reviews_int[i] = [0]*(acceptable_len - len(rev)) + rev

print(reviews_int[:3])


[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [170]:
# Step 5. Define Dataset and Dataloader

class ReviewDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, index):
        return self.X[index], self.y[index]

# Convert positive/negative to 1/0 labels
feedback_list = [1 if label == 'positive' else 0 for label in df.iloc[:, 1]]

# Split train and test data 
split_fraction = 0.8
feature_len = len(reviews_int)

X_train = reviews_int[0:int(split_fraction*feature_len)]
X_test = reviews_int[int(split_fraction*feature_len):]
y_train = feedback_list[0:int(split_fraction*feature_len)]
y_test = feedback_list[int(split_fraction*feature_len):]

# len(X_train), len(X_test)

X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_dataset = ReviewDataset(X_train, y_train)
test_dataset = ReviewDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)


In [171]:
# Step 6: LSTM model 

class ReviewLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim, 
            hidden_size=hidden_dim, 
            num_layers=1, 
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        intermediate_hidden_state, (final_hidden_state, final_cell_state) = self.lstm(embedded)
        # Use last hidden state of the last LSTM layer
        output = self.fc(final_hidden_state[-1].squeeze(0))
        # output = self.sigmoid(output)
        return output

embedding_dim = 64 
hidden_dim = 256
vocab_size = len(vocab)

model = ReviewLSTM(vocab_size, embedding_dim, hidden_dim)

In [ ]:
# Step 7. Training code based on epoch and batch size, calculate loss 

epochs = 10
lr = 0.001
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

for i in range(epochs):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        output = model(batch_x).squeeze(1)
        # print('Comparing shapes: ', output.shape, batch_y.shape)
        loss = criterion(output, batch_y.float())
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch: {i+1} Loss: {total_loss:.4f}")

Epoch: 1 Loss: 103.1547


KeyboardInterrupt: 

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab, 
    'acceptable_len': acceptable_len
}, 'imdb_sentiment_model.pth')

In [ ]:
# Predict function 

model.eval() 

def prediction(model, vocab, text):
    tokenized_text = word_tokenize(text.lower())
    
    # Text to numerical indices 
    numerical_text = []
    for word in tokenized_text:
        numerical_text.append(word_to_num(word, vocab))

    padded_text = [0]*(acceptable_len - len(numerical_text)) + numerical_text
    padded_text = torch.tensor(padded_text, dtype=torch.long).unsqueeze(0)

    with torch.no_grad():
        output = model(padded_text)
        predict_val = torch.sigmoid(output).squeeze(0).item()
    # print(output)   
    
    # Predicted index 
    # value, index = torch.max(output, dim=1)
    # print(value, index)

    return predict_val

In [ ]:
input_texts = [
    "The movie really sucked!",
    "I liked the movie",
    "Very good acting",
    "I'd rate a 9 out of 10",
    "Totally disappointed. I'll rate a 1 star.",
    "It was horrible. Very poor cast.",
    "It was a total waste of time. Way below expectation.",
    "Awesome film! Thanks a lot."
]

for input_text in input_texts:
    output_text = prediction(model, vocab, input_text)
    print(f"{input_text} -> {output_text}")

The movie really sucked! -> 0.7985125780105591
I liked the movie -> 0.8993810415267944
Very good acting -> 0.8708845376968384
I'd rate a 9 out of 10 -> 0.8816401958465576
Totally disappointed. I'll rate a 1 star. -> 0.5084652304649353
It was horrible. Very poor cast. -> 0.41367167234420776
It was a total waste of time. Way below expectation. -> 0.6004467010498047
Awesome film! Thanks a lot. -> 0.6473555564880371
